In [2]:
import pandas as pd

# Load the original intent dataset
intent_df = pd.read_csv("amazon_intent_base.csv")

# Random 1,000 queries
sample_1000 = (
    intent_df[
        ["customer_tweet_id", "clean_customer_query"]
    ]
    .dropna(subset=["clean_customer_query"])
    .sample(n=1000, random_state=42)
    .reset_index(drop=True)
)

print("Total queries selected:", len(sample_1000))
print("\nFirst 10 queries:")
print(sample_1000.head(10).to_string(index=False))

Total queries selected: 1000

First 10 queries:
 customer_tweet_id                                                                                                                                                                                          clean_customer_query
          594532.0                                                                                                                                                                                                           UPS
          588782.0 I just got a package (a birthday gift for tomorrow) and the packaging and the item itself smell very strongly of smoke. I'm assuming the driver at Intelcom was smoking in the car and now my gift is ruined.
         1704079.0                                                                                             Once again, I'm not getting my packages on time from #Intelcom. So much for 2 day delivery! This is the 5th time!
          831799.0                                  

In [8]:
# The original content of this cell was '!ollama pull qwen3:4b'.
# If you need to pull the qwen3:4b model again, please uncomment and run the line below:
# !ollama pull qwen3:4b

# To push files to Git from Google Colab, you generally need to perform the following steps.
# Please note that pushing to Git usually requires authentication and careful handling of credentials.
# Replace <YOUR_USERNAME>, <YOUR_EMAIL>, <YOUR_REPO_NAME>, and <YOUR_BRANCH> with your actual details.

# 1. Configure Git user name and email (if not already done in the session):
# !git config user.name "Your Name"
# !git config user.email "your.email@example.com"

# 2. Initialize a Git repository in your /content directory if it's not already one.
#    (Be careful with this step if your /content is already a git repo, or linked to one)
# !git init

# 3. Add the files you want to track. Use '.' to add all files in the current directory or specify files.
# !git add .
# For example:
# !git add amazon_discovery_results.jsonl amazon_candidate_taxonomy.json amazon_sample_1000.csv

# 4. Commit the changes:
# !git commit -m "Add Colab generated files"

# 5. Add the remote repository URL (if you haven't already linked to your repo):
#    You might need to use a Personal Access Token (PAT) instead of your password for GitHub or other services.
# !git remote add origin https://github.com/<YOUR_USERNAME>/<YOUR_REPO_NAME>.git

# 6. Push the changes to your remote repository.
# !git push -u origin <YOUR_BRANCH>
# If prompted, use your GitHub username and a Personal Access Token (PAT) as the password.

In [6]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "qwen3:4b"

def ask_ollama_json(prompt):
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "think": False,
        "format": "json",

        # Disable model/context caching after each request
        "keep_alive": 0,

        "options": {
            "temperature": 0,
            "num_predict": 600
        }
    }

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=300
    )

    response.raise_for_status()

    return json.loads(response.json()["response"])

In [7]:
import json
import os
import gc
import time
import pandas as pd

# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

BATCH_SIZE = 10

INPUT_FILE = "amazon_intent_base.csv"
SAMPLE_FILE = "amazon_sample_1000.csv"
RESULT_FILE = "amazon_discovery_results.jsonl"
TAXONOMY_FILE = "amazon_candidate_taxonomy.json"

# --------------------------------------------------
# LOAD ONLY WHAT WE NEED
# --------------------------------------------------

intent_df = pd.read_csv(
    INPUT_FILE,
    usecols=["customer_tweet_id", "clean_customer_query"]
)

# Create the 1000-query sample only once
if os.path.exists(SAMPLE_FILE):

    sample_1000 = pd.read_csv(SAMPLE_FILE)

    print("Loaded existing 1000-query sample.")

else:

    sample_1000 = (
        intent_df[
            ["customer_tweet_id", "clean_customer_query"]
        ]
        .dropna(subset=["clean_customer_query"])
        .sample(n=1000, random_state=42)
        .reset_index(drop=True)
    )

    sample_1000.to_csv(
        SAMPLE_FILE,
        index=False
    )

    print("Created and saved 1000-query sample.")

# We no longer need the full dataset in memory
del intent_df
gc.collect()

print("Queries:", len(sample_1000))


# --------------------------------------------------
# LOAD OR INITIALIZE TAXONOMY
# --------------------------------------------------

if os.path.exists(TAXONOMY_FILE):

    with open(TAXONOMY_FILE, "r", encoding="utf-8") as f:
        discovered_intents = json.load(f)

    print("Loaded existing taxonomy.")

else:

    discovered_intents = [
        {
            "name": "Non-Informative",
            "definition": (
                "A message containing no meaningful customer-support "
                "request, problem, or actionable information."
            )
        }
    ]

    with open(TAXONOMY_FILE, "w", encoding="utf-8") as f:
        json.dump(
            discovered_intents,
            f,
            indent=2,
            ensure_ascii=False
        )


# --------------------------------------------------
# FIND ALREADY PROCESSED BATCHES
# --------------------------------------------------

processed_ids = set()

if os.path.exists(RESULT_FILE):

    with open(RESULT_FILE, "r", encoding="utf-8") as f:

        for line in f:

            try:
                record = json.loads(line)

                processed_ids.add(
                    str(record["customer_tweet_id"])
                )

            except:
                pass

print("Already processed:", len(processed_ids))


# --------------------------------------------------
# DISCOVERY PROMPT
# --------------------------------------------------

def build_discovery_prompt(batch, current_intents):

    intent_text = "\n".join(
        f"- {x['name']}: {x['definition']}"
        for x in current_intents
    )

    query_text = "\n".join(
        f"{i+1}. ID={row.customer_tweet_id} | "
        f"{row.clean_customer_query}"
        for i, row in enumerate(
            batch.itertuples(index=False)
        )
    )

    return f"""
You are designing an intent taxonomy for Amazon customer support.

CURRENT INTENTS:

{intent_text}

CUSTOMER QUERIES:

{query_text}

Assign exactly ONE intent to every query.

RULES:

1. Reuse an existing intent ONLY when it genuinely represents
   the same underlying customer problem.

2. NEVER force a meaningful customer problem into an unrelated
   existing intent.

3. If no existing intent genuinely fits, create a new meaningful
   intent.

4. New intents should represent customer-support problems, not
   individual products, names, countries, or wording variations.

5. Keep intents reasonably broad.

6. Non-Informative is ONLY for messages containing no meaningful
   support request, problem, or actionable information.

7. Do NOT merge, rename, or delete existing intents.

8. Return exactly one label for every query.

9. Only include genuinely NEW intents in new_intents.

Return ONLY valid JSON:

{{
  "new_intents": [
    {{
      "name": "Intent Name",
      "definition": "Short definition"
    }}
  ],
  "labels": [
    {{
      "id": "customer tweet id",
      "intent": "Intent Name"
    }}
  ]
}}
"""


# --------------------------------------------------
# PROCESS BATCHES
# --------------------------------------------------

total = len(sample_1000)

for start in range(0, total, BATCH_SIZE):

    batch = sample_1000.iloc[
        start:start + BATCH_SIZE
    ].copy()

    # Skip queries already processed
    batch = batch[
        ~batch["customer_tweet_id"]
        .astype(str)
        .isin(processed_ids)
    ]

    if len(batch) == 0:
        continue

    print(
        f"\nProcessing queries "
        f"{start + 1}-{min(start + BATCH_SIZE, total)}..."
    )

    prompt = build_discovery_prompt(
        batch,
        discovered_intents
    )

    try:

        result = ask_ollama_json(prompt)

        # ------------------------------------------
        # ADD NEW INTENTS
        # ------------------------------------------

        existing_names = {
            x["name"].lower()
            for x in discovered_intents
        }

        for new_intent in result.get(
            "new_intents",
            []
        ):

            name = new_intent["name"].strip()

            if name.lower() not in existing_names:

                discovered_intents.append({
                    "name": name,
                    "definition": (
                        new_intent["definition"]
                        .strip()
                    )
                })

                existing_names.add(
                    name.lower()
                )

        # ------------------------------------------
        # WRITE RESULTS IMMEDIATELY TO DISK
        # ------------------------------------------

        with open(
            RESULT_FILE,
            "a",
            encoding="utf-8"
        ) as f:

            for label in result.get(
                "labels",
                []
            ):

                record = {
                    "customer_tweet_id":
                        str(label["id"]),

                    "intent":
                        label["intent"].strip()
                }

                f.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    ) + "\n"
                )

                processed_ids.add(
                    str(label["id"])
                )

        # ------------------------------------------
        # SAVE TAXONOMY
        # ------------------------------------------

        with open(
            TAXONOMY_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                discovered_intents,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            "Processed:",
            len(processed_ids),
            "| Intents:",
            len(discovered_intents)
        )

    except Exception as e:

        print(
            "Batch failed:",
            start,
            "-",
            start + BATCH_SIZE,
            "|",
            repr(e)
        )

    # ------------------------------------------
    # RELEASE MEMORY
    # ------------------------------------------

    del batch
    del prompt

    if "result" in locals():
        del result

    gc.collect()

    # Small pause so the system can recover
    time.sleep(0.5)


print("\n==============================")
print("DISCOVERY FINISHED")
print("==============================")
print("Queries processed:", len(processed_ids))
print("Candidate intents:", len(discovered_intents))
print("Results saved to:", RESULT_FILE)
print("Taxonomy saved to:", TAXONOMY_FILE)

Loaded existing 1000-query sample.
Queries: 1000
Loaded existing taxonomy.
Already processed: 0

Processing queries 1-10...
Processed: 10 | Intents: 9

Processing queries 11-20...
Processed: 20 | Intents: 10

Processing queries 21-30...
Processed: 30 | Intents: 11

Processing queries 31-40...
Processed: 40 | Intents: 12

Processing queries 41-50...
Processed: 50 | Intents: 13

Processing queries 51-60...
Processed: 60 | Intents: 14

Processing queries 61-70...
Processed: 70 | Intents: 14

Processing queries 71-80...
Processed: 80 | Intents: 14

Processing queries 81-90...
Processed: 90 | Intents: 15

Processing queries 91-100...
Processed: 100 | Intents: 16

Processing queries 101-110...
Processed: 110 | Intents: 17

Processing queries 111-120...
Processed: 120 | Intents: 17

Processing queries 121-130...
Processed: 129 | Intents: 18

Processing queries 131-140...
Processed: 139 | Intents: 18

Processing queries 141-150...
Processed: 149 | Intents: 18

Processing queries 151-160...
Pro

In [9]:
from sentence_transformers import SentenceTransformer
import json
import numpy as np

# Load candidate taxonomy
with open(
    "amazon_candidate_taxonomy.json",
    "r",
    encoding="utf-8"
) as f:
    candidate_taxonomy = json.load(f)

print("Candidate labels:", len(candidate_taxonomy))

# Load embedding model
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

# Build semantic representation of each label
label_texts = [
    f"{intent['name']}: {intent['definition']}"
    for intent in candidate_taxonomy
]

# Generate label embeddings
label_embeddings = model.encode(
    label_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", label_embeddings.shape)

Candidate labels: 53


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (53, 384)


In [10]:
np.save(
    "amazon_label_embeddings.npy",
    label_embeddings
)

print("Saved label embeddings.")

Saved label embeddings.


In [11]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# Load saved label embeddings
label_embeddings = np.load(
    "amazon_label_embeddings.npy"
)

# Load taxonomy
with open(
    "amazon_candidate_taxonomy.json",
    "r",
    encoding="utf-8"
) as f:
    candidate_taxonomy = json.load(f)

label_names = [
    x["name"]
    for x in candidate_taxonomy
]

# Pairwise cosine similarity
similarity_matrix = cosine_similarity(
    label_embeddings
)

# Collect only unique pairs
pairs = []

for i in range(len(label_names)):

    for j in range(i + 1, len(label_names)):

        pairs.append({
            "label_1": label_names[i],
            "label_2": label_names[j],
            "similarity": similarity_matrix[i, j]
        })

similarity_df = pd.DataFrame(pairs)

# Highest similarity first
similarity_df = similarity_df.sort_values(
    "similarity",
    ascending=False
).reset_index(drop=True)

print("Total label pairs:", len(similarity_df))

Total label pairs: 1378


In [12]:
pd.set_option("display.max_rows", 100)

similarity_df.head(40)

,label_1,label_2,similarity
0,Refund Processing Delay,Refund Request Delay,0.967702
1,Delivery Status Confusion,Delivery Confusion,0.958952
2,Delivery Status Miscommunication,Delivery Status Inconsistency,0.957277
3,Refund Request,Refund Request Due to Unauthorized Transaction,0.953944
4,Account Blockage Without Reason,Account Blockage Without Verification,0.949078
5,Urgent Formula Shortage Concern,Urgent Product Shortage Concern,0.932413
6,Account Blockage Due to Fraud Concerns,Account Blockage Without Verification,0.929572
7,Delivery Status Uncertainty,Delivery Confusion,0.926309
8,Delivery Confusion,Delivery Status Miscommunication,0.909622
9,Missing Delivery Contact,Delivery Contact Request,0.905876


In [15]:
import json
import pandas as pd

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

MERGE_THRESHOLD = 0.90

RESULT_FILE = "amazon_discovery_results.jsonl"
TAXONOMY_FILE = "amazon_candidate_taxonomy.json"

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

with open(TAXONOMY_FILE, "r", encoding="utf-8") as f:
    candidate_taxonomy = json.load(f)

discovery_labels = []

with open(RESULT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            discovery_labels.append(json.loads(line))

labels_df = pd.DataFrame(discovery_labels)

# ------------------------------------------------------------
# STRONG SIMILARITY PAIRS
# ------------------------------------------------------------

strong_pairs = similarity_df[
    similarity_df["similarity"] >= MERGE_THRESHOLD
].copy()

print("Strong pairs:", len(strong_pairs))

# ------------------------------------------------------------
# CREATE MERGE MAP
# ------------------------------------------------------------

merge_map = {}

for _, row in strong_pairs.iterrows():

    label_a = row["label_1"]
    label_b = row["label_2"]

    # If neither has been assigned, keep label_a
    if label_a not in merge_map and label_b not in merge_map:

        merge_map[label_b] = label_a

    # If B already maps somewhere, map A's group to B's group
    elif label_a not in merge_map:

        merge_map[label_a] = merge_map.get(
            label_b,
            label_b
        )

    elif label_b not in merge_map:

        merge_map[label_b] = merge_map.get(
            label_a,
            label_a
        )

# Resolve chains
def resolve_label(label):

    visited = set()

    while label in merge_map and label not in visited:

        visited.add(label)
        label = merge_map[label]

    return label


merge_map = {
    label: resolve_label(label)
    for label in merge_map
}

# ------------------------------------------------------------
# APPLY MERGES TO DISCOVERY RESULTS
# ------------------------------------------------------------

labels_df["original_intent"] = labels_df["intent"]

labels_df["intent"] = labels_df["intent"].apply(
    resolve_label
)

# ------------------------------------------------------------
# BUILD NEW TAXONOMY
# ------------------------------------------------------------

new_taxonomy = []

for intent in candidate_taxonomy:

    original_name = intent["name"]

    # Skip labels that were merged into another label
    if original_name in merge_map:

        continue

    new_taxonomy.append({
        "name": original_name,
        "definition": intent["definition"]
    })

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

with open(
    "amazon_minimized_taxonomy_v1.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        new_taxonomy,
        f,
        indent=2,
        ensure_ascii=False
    )

labels_df.to_json(
    "amazon_discovery_labels_v1.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\n==============================")
print("MERGE COMPLETE")
print("==============================")

print(
    "Original labels:",
    len(candidate_taxonomy)
)

print(
    "Labels after strong merges:",
    len(new_taxonomy)
)

print(
    "Labels removed:",
    len(candidate_taxonomy) - len(new_taxonomy)
)

print("\nMerge map:")

for old, new in merge_map.items():

    print(
        f"  {old}  -->  {new}"
    )

Strong pairs: 13

MERGE COMPLETE
Original labels: 53
Labels after strong merges: 40
Labels removed: 13

Merge map:
  Refund Request Delay  -->  Refund Processing Delay
  Delivery Confusion  -->  Delivery Status Confusion
  Delivery Status Inconsistency  -->  Delivery Status Confusion
  Refund Request Due to Unauthorized Transaction  -->  Refund Request
  Account Blockage Without Verification  -->  Account Blockage Without Reason
  Urgent Product Shortage Concern  -->  Urgent Formula Shortage Concern
  Account Blockage Due to Fraud Concerns  -->  Account Blockage Without Reason
  Delivery Status Uncertainty  -->  Delivery Status Confusion
  Delivery Status Miscommunication  -->  Delivery Status Confusion
  Delivery Contact Request  -->  Missing Delivery Contact
  Price Discrepancy  -->  Payment Discrepancy
  Delivery Status Stalemate  -->  Delivery Status Confusion
  Account Blockage Without Reason  -->  Account Blockage Without Reason


In [16]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# Load minimized taxonomy
with open(
    "amazon_minimized_taxonomy_v1.json",
    "r",
    encoding="utf-8"
) as f:
    minimized_taxonomy = json.load(f)

print("Labels remaining:", len(minimized_taxonomy))

# Load embedding model
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

# Embed label name + definition
label_texts_v1 = [
    f"{intent['name']}: {intent['definition']}"
    for intent in minimized_taxonomy
]

label_embeddings_v1 = model.encode(
    label_texts_v1,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(
    "Embedding shape:",
    label_embeddings_v1.shape
)

# Save
np.save(
    "amazon_label_embeddings_v1.npy",
    label_embeddings_v1
)

print("Saved amazon_label_embeddings_v1.npy")

Labels remaining: 40


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (40, 384)
Saved amazon_label_embeddings_v1.npy


In [17]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

label_embeddings_v1 = np.load(
    "amazon_label_embeddings_v1.npy"
)

with open(
    "amazon_minimized_taxonomy_v1.json",
    "r",
    encoding="utf-8"
) as f:
    minimized_taxonomy = json.load(f)

label_names_v1 = [
    x["name"]
    for x in minimized_taxonomy
]

# ------------------------------------------------------------
# PAIRWISE SIMILARITY
# ------------------------------------------------------------

similarity_matrix_v1 = cosine_similarity(
    label_embeddings_v1
)

pairs_v1 = []

for i in range(len(label_names_v1)):

    for j in range(i + 1, len(label_names_v1)):

        pairs_v1.append({
            "label_1": label_names_v1[i],
            "label_2": label_names_v1[j],
            "similarity": similarity_matrix_v1[i, j]
        })

similarity_df_v1 = (
    pd.DataFrame(pairs_v1)
    .sort_values(
        "similarity",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# STATISTICS
# ------------------------------------------------------------

print("Labels:", len(label_names_v1))
print("Total pairs:", len(similarity_df_v1))

print("\nSimilarity statistics:")
print(
    similarity_df_v1["similarity"].describe()
)

print("\nPairs >= 0.90:",
      (similarity_df_v1["similarity"] >= 0.90).sum())

print("Pairs >= 0.85:",
      (similarity_df_v1["similarity"] >= 0.85).sum())

print("Pairs >= 0.80:",
      (similarity_df_v1["similarity"] >= 0.80).sum())

# ------------------------------------------------------------
# TOP STRONG CANDIDATES
# ------------------------------------------------------------

print("\nTop similarity pairs:")
print(
    similarity_df_v1.head(30).to_string(
        index=False
    )
)

Labels: 40
Total pairs: 780

Similarity statistics:
count    780.000000
mean       0.673701
std        0.067426
min        0.506111
25%        0.628677
50%        0.671207
75%        0.717307
max        0.891426
Name: similarity, dtype: float64

Pairs >= 0.90: 0
Pairs >= 0.85: 9
Pairs >= 0.80: 28

Top similarity pairs:
                      label_1                             label_2  similarity
        Delivery Report Issue           Delivery Status Confusion    0.891426
Customer Care Ineffectiveness           Recurring Service Failure    0.876262
   Courier Unethical Behavior                   Courier Complaint    0.875593
      Refund Processing Delay                  Refund Discrepancy    0.874148
    Delivery Status Confusion   Delivery Status Misrepresentation    0.868804
               Delivery Delay           Delivery Status Confusion    0.867173
          Payment Discrepancy                  Refund Discrepancy    0.864266
          Payment Discrepancy             Double Charge

In [18]:
import json
import os
import gc
import time
import pandas as pd

# ------------------------------------------------------------
# FILES
# ------------------------------------------------------------

SAMPLE_FILE = "amazon_sample_1000.csv"
TAXONOMY_FILE = "amazon_minimized_taxonomy_v1.json"

OUTPUT_FILE = "amazon_second_pass_labels.jsonl"

# ------------------------------------------------------------
# LOAD TAXONOMY
# ------------------------------------------------------------

with open(
    TAXONOMY_FILE,
    "r",
    encoding="utf-8"
) as f:
    final_taxonomy = json.load(f)

print("Available intents:", len(final_taxonomy))

for i, intent in enumerate(final_taxonomy, 1):
    print(
        f"{i}. {intent['name']}: "
        f"{intent['definition']}"
    )

# ------------------------------------------------------------
# LOAD SAMPLE
# ------------------------------------------------------------

sample_1000 = pd.read_csv(
    SAMPLE_FILE,
    usecols=[
        "customer_tweet_id",
        "clean_customer_query"
    ]
)

sample_1000["customer_tweet_id"] = (
    sample_1000["customer_tweet_id"]
    .astype(str)
)

print("\nQueries:", len(sample_1000))

# ------------------------------------------------------------
# FIND ALREADY PROCESSED
# ------------------------------------------------------------

processed_ids = set()

if os.path.exists(OUTPUT_FILE):

    with open(
        OUTPUT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if line.strip():

                try:
                    record = json.loads(line)

                    processed_ids.add(
                        str(record["customer_tweet_id"])
                    )

                except:
                    pass

print(
    "Already processed:",
    len(processed_ids)
)

gc.collect()

Available intents: 40
1. Non-Informative: A message containing no meaningful customer-support request, problem, or actionable information.
2. Delivery Delay: Customer reports repeated delays in package delivery
3. Package Smell Complaint: Customer reports a package smelling strongly of smoke or other unusual odors
4. Courier Unethical Behavior: Customer reports unethical behavior by courier partners
5. Delivery Report Issue: Customer reports issues with delivery confirmation reports
6. Customer Care Ineffectiveness: Customer reports that customer care has not resolved their issue
7. Money Update Request: Customer requests updates on money related to fraud or payment issues
8. Price Increase After Reorder: Customer reports being asked to reorder at an increased price
9. Missing Delivery Contact: Customer requests delivery contact information from courier
10. Unresolved Complaint: Customer reports that their issue has not been resolved despite previous efforts
11. Refund Request: Custome

3973

In [19]:
BATCH_SIZE = 10

def build_closed_set_prompt(batch, taxonomy):

    taxonomy_text = "\n".join(
        f"{i+1}. {x['name']}: {x['definition']}"
        for i, x in enumerate(taxonomy)
    )

    query_text = "\n".join(
        f"{i+1}. ID={row.customer_tweet_id} | "
        f"{row.clean_customer_query}"
        for i, row in enumerate(
            batch.itertuples(index=False)
        )
    )

    return f"""
You are classifying Amazon customer-support queries.

You MUST choose exactly ONE intent from the existing taxonomy.

IMPORTANT:
- You CANNOT create a new intent.
- You CANNOT rename an intent.
- You CANNOT merge intents.
- Use the intent whose underlying customer problem best matches.
- Do not classify a meaningful support problem as Non-Informative.
- Non-Informative is only for messages with no meaningful
  customer-support request, problem, or actionable information.

EXISTING TAXONOMY:

{taxonomy_text}

CUSTOMER QUERIES:

{query_text}

Return ONLY valid JSON:

{{
  "labels": [
    {{
      "id": "customer tweet id",
      "intent": "EXACT intent name"
    }}
  ]
}}

Return exactly one label for every query.
"""


total = len(sample_1000)

for start in range(0, total, BATCH_SIZE):

    batch = sample_1000.iloc[
        start:start + BATCH_SIZE
    ].copy()

    # Skip completed queries
    batch = batch[
        ~batch["customer_tweet_id"].isin(
            processed_ids
        )
    ]

    if len(batch) == 0:
        continue

    print(
        f"Processing "
        f"{start + 1}-{min(start + BATCH_SIZE, total)}..."
    )

    prompt = build_closed_set_prompt(
        batch,
        final_taxonomy
    )

    try:

        result = ask_ollama_json(prompt)

        labels = result.get(
            "labels",
            []
        )

        # Write immediately to disk
        with open(
            OUTPUT_FILE,
            "a",
            encoding="utf-8"
        ) as f:

            for label in labels:

                record = {
                    "customer_tweet_id":
                        str(label["id"]),

                    "intent":
                        label["intent"].strip()
                }

                f.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    ) + "\n"
                )

                processed_ids.add(
                    str(label["id"])
                )

        print(
            "Processed:",
            len(processed_ids),
            "/",
            total
        )

    except Exception as e:

        print(
            "Batch failed:",
            start,
            repr(e)
        )

    # Free memory
    del batch
    del prompt

    if "result" in locals():
        del result

    gc.collect()

    time.sleep(0.5)


print("\n==============================")
print("SECOND PASS COMPLETE")
print("==============================")
print(
    "Queries labeled:",
    len(processed_ids)
)
print(
    "Taxonomy size:",
    len(final_taxonomy)
)
print(
    "Saved:",
    OUTPUT_FILE
)

Processing 1-10...
Processed: 10 / 1000
Processing 11-20...
Processed: 17 / 1000
Processing 21-30...
Processed: 27 / 1000
Processing 31-40...
Processed: 37 / 1000
Processing 41-50...
Processed: 47 / 1000
Processing 51-60...
Processed: 57 / 1000
Processing 61-70...
Processed: 67 / 1000
Processing 71-80...
Processed: 77 / 1000
Processing 81-90...
Processed: 87 / 1000
Processing 91-100...
Processed: 97 / 1000
Processing 101-110...
Processed: 107 / 1000
Processing 111-120...
Processed: 117 / 1000
Processing 121-130...
Processed: 126 / 1000
Processing 131-140...
Processed: 136 / 1000
Processing 141-150...
Processed: 145 / 1000
Processing 151-160...
Processed: 155 / 1000
Processing 161-170...
Processed: 163 / 1000
Processing 171-180...
Processed: 173 / 1000
Processing 181-190...
Processed: 183 / 1000
Processing 191-200...
Processed: 193 / 1000
Processing 201-210...
Processed: 194 / 1000
Processing 211-220...
Processed: 204 / 1000
Processing 221-230...
Processed: 214 / 1000
Processing 231-240

In [20]:
import os
import shutil
import json

# Create a project checkpoint folder
CHECKPOINT_DIR = "amazon_intent_project_checkpoint"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

files_to_save = [
    "amazon_intent_base.csv",
    "amazon_sample_1000.csv",
    "amazon_discovery_results.jsonl",
    "amazon_candidate_taxonomy.json",
    "amazon_query_embeddings.npy",
    "amazon_label_embeddings.npy",
    "amazon_minimized_taxonomy_v1.json",
    "amazon_label_embeddings_v1.npy",
    "amazon_second_pass_labels.jsonl"
]

saved = []

for file in files_to_save:

    if os.path.exists(file):

        shutil.copy2(
            file,
            os.path.join(
                CHECKPOINT_DIR,
                file
            )
        )

        saved.append(file)

print("================================")
print("CHECKPOINT SAVED")
print("================================")

for file in saved:
    print("✓", file)

print(
    f"\nSaved {len(saved)} artifacts."
)

CHECKPOINT SAVED
✓ amazon_intent_base.csv
✓ amazon_sample_1000.csv
✓ amazon_discovery_results.jsonl
✓ amazon_candidate_taxonomy.json
✓ amazon_label_embeddings.npy
✓ amazon_minimized_taxonomy_v1.json
✓ amazon_label_embeddings_v1.npy
✓ amazon_second_pass_labels.jsonl

Saved 8 artifacts.


In [21]:
manifest = {
    "project": "Amazon Customer Support Intent Classification",

    "dataset": "amazon_intent_base.csv",

    "discovery": {
        "sample_size": 1000,
        "processed": 991,
        "candidate_intents": 53
    },

    "embedding_model": "BAAI/bge-small-en-v1.5",

    "compression": {
        "method": "label-name-plus-definition embeddings",
        "similarity_metric": "cosine similarity",
        "strong_merge_threshold": 0.90,
        "initial_intents": 53,
        "remaining_intents": 40
    },

    "second_pass": {
        "taxonomy_size": 40,
        "labeled_queries": 972,
        "unlabeled_queries": 28
    },

    "next_step": "Label full amazon_intent_base.csv using frozen 40-intent taxonomy"
}

with open(
    os.path.join(
        CHECKPOINT_DIR,
        "project_manifest.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

print("✓ project_manifest.json saved")

✓ project_manifest.json saved
